In [ ]:
print("Họ tên: Lâm Đặng Gia Kiện - MSSV: 6351071036")
print("Câu 6 - Bài 1 - Phần 1")
import re
import numpy as np

print("\n========== CÂU 6: TIỀN XỬ LÝ DỮ LIỆU ==========")
# 1. Trích xuất đặc trưng Deck từ Cabin
deck = {"A": 1, "B": 2, "C": 3, "D": 4, "E": 5, "F": 6, "G": 7, "U": 8}
if 'cabin' in df.columns:
    df['cabin'] = df['cabin'].fillna("U0")
    df['deck'] = df['cabin'].map(lambda x: re.compile("([a-zA-Z]+)").search(x).group())
    df['deck'] = df['deck'].map(deck).fillna(0).astype(int)

# Khảo sát cột ticket trước khi xóa để đưa vào báo cáo
print("--- Thống kê thuộc tính 'ticket' trước khi xóa ---")
if 'ticket' in df.columns:
    print(df['ticket'].describe())

# Thực hiện xóa các thuộc tính không cần thiết
cols_to_drop = ['cabin', 'ticket', 'boat', 'body', 'home.dest']
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])
print(f"\n[+] Đã xóa các cột không cần thiết: {cols_to_drop}")

# 2. Điền giá trị còn thiếu vào thuộc tính tuổi (age)
print(f"\n--- Số lượng NaN của cột 'age' trước khi xử lý: {df['age'].isnull().sum()}")
if df['age'].isnull().sum() > 0:
    mean, std = df["age"].mean(), df["age"].std()
    is_null = df["age"].isnull().sum()
    rand_age = np.random.randint(mean - std, mean + std, size=is_null)
    age_slice = df["age"].copy()
    age_slice[np.isnan(age_slice)] = rand_age
    df["age"] = age_slice.astype(int)
print(f"--- Số lượng NaN của cột 'age' sau khi xử lý bằng random: {df['age'].isnull().sum()}")

# 3. Điền giá trị còn thiếu vào Cổng lên tàu (embarked)
print("\n--- Thống kê thuộc tính 'embarked' trước khi xử lý ---")
print(df['embarked'].describe())
df['embarked'] = df['embarked'].fillna('S')


print("\n========== CÂU 7: XEM XÉT LẠI CÁC THUỘC TÍNH ==========")
print("--- Cấu trúc dữ liệu sau khi xóa cột và điền dữ liệu rỗng ---")
df.info()
print("\n")

# Chuyển kiểu dữ liệu giá vé
df['fare'] = df['fare'].fillna(0).astype(int)

# Trích xuất danh hiệu
if 'name' in df.columns:
    titles = {"Mr": 1, "Miss": 2, "Mrs": 3, "Master": 4, "Rare": 5}
    df['title'] = df['name'].str.extract(' ([A-Za-zA-Z]+)\.', expand=False)
    df['title'] = df['title'].replace(['Lady', 'Countess','Capt', 'Col','Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare')
    df['title'] = df['title'].replace(['Mlle', 'Ms'], 'Miss')
    df['title'] = df['title'].replace('Mme', 'Mrs')
    df['title'] = df['title'].map(titles).fillna(0)
    df = df.drop(['name'], axis=1)

# Biểu diễn số các cột hạng mục
if df['sex'].dtype == 'O':
    df['sex'] = df['sex'].map({"male": 0, "female": 1})

if df['embarked'].dtype == 'O':
    df['embarked'] = df['embarked'].map({"S": 0, "C": 1, "Q": 2})

# Rời rạc hóa cột tuổi (Age)
df['age'] = df['age'].astype(int)
df.loc[df['age'] <= 11, 'age'] = 0
df.loc[(df['age'] > 11) & (df['age'] <= 18), 'age'] = 1
df.loc[(df['age'] > 18) & (df['age'] <= 22), 'age'] = 2
df.loc[(df['age'] > 22) & (df['age'] <= 27), 'age'] = 3
df.loc[(df['age'] > 27) & (df['age'] <= 33), 'age'] = 4
df.loc[(df['age'] > 33) & (df['age'] <= 40), 'age'] = 5
df.loc[(df['age'] > 40) & (df['age'] <= 66), 'age'] = 6
df.loc[df['age'] > 66, 'age'] = 7

print("--- Kết quả phân phối số lượng hành khách theo nhóm tuổi (age) ---")
print(df['age'].value_counts())

# Rời rạc hóa cột giá vé (Fare)
df.loc[df['fare'] <= 7.91, 'fare'] = 0
df.loc[(df['fare'] > 7.91) & (df['fare'] <= 14.454), 'fare'] = 1
df.loc[(df['fare'] > 14.454) & (df['fare'] <= 31), 'fare'] = 2
df.loc[(df['fare'] > 31) & (df['fare'] <= 99), 'fare'] = 3
df.loc[(df['fare'] > 99) & (df['fare'] <= 250), 'fare'] = 4
df.loc[df['fare'] > 250, 'fare'] = 5
df['fare'] = df['fare'].astype(int)


print("\n========== CÂU 8: TẠO THUỘC TÍNH MỚI ==========")
df['age_class'] = df['age'] * df['pclass']
df['fare_per_person'] = (df['fare'] / (df['relatives'] + 1)).astype(int)

print("--- 10 dòng đầu tiên của tập dữ liệu sau khi hoàn tất tiền xử lý ---")
display(df.head(10))